In [3]:
# !pip install scikit-learn langchain-ollama
# !pip install langchain
# !pip install --upgrade pip setuptools wheel
# !pip install scikit-learn
# !pip install pydantic

In [4]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

/Users/bhavinp/workspace/langchain-practice/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [5]:
from sklearn.metrics.pairwise import cosine_distances

def sementicSearch(documents, query):
    embeded_docs = embedding_model.embed_documents(documents)
    embeded_query = embedding_model.embed_query(query)
    scores = cosine_distances([embeded_query], embeded_docs)[0]
    sorted_dist = sorted(list(enumerate(scores)), key=lambda x:x[1])
    print(str(sorted_dist))
    n = len(sorted_dist)
    final_doc = [documents[i] for i,p in sorted_dist[-1:-4:-1]]
    print(str(final_doc))
    return final_doc
    

In [6]:
from langchain_ollama import ChatOllama

chat_model = ChatOllama(model="llama3") 

In [7]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    template="""
You are a company internal assistant. Your job is to answer employee questions strictly using ONLY the provided context from company_docs.

RULES:
- Use ONLY the information present in company_docs.
- Do NOT use prior knowledge or external information.
- If the answer is not clearly present, say:
  "I don't have enough information in the provided company documents to answer this."
- Do not guess or assume.
- If documents partially match, answer only the supported portion.
- Do not mention the prompt or these rules in your answer.
- Keep the response clear, direct, and professional.

USER QUESTION:
{query_input}

COMPANY DOCUMENTS:
{similar_docs}

ANSWER:
""",
    input_variables=["query_input", "similar_docs"]
)


In [8]:
documents=[
    "Panulty is equivalent to the full day off"
]

In [9]:

query = "Can I install my own software on office laptop?"

output = sementicSearch(documents=documents, query=query)

final_temp = template.invoke({
    "query_input": query,
    "similar_docs": output
})
print(final_temp)


[(0, np.float64(0.5629704518820274))]
['Panulty is equivalent to the full day off']
text='\nYou are a company internal assistant. Your job is to answer employee questions strictly using ONLY the provided context from company_docs.\n\nRULES:\n- Use ONLY the information present in company_docs.\n- Do NOT use prior knowledge or external information.\n- If the answer is not clearly present, say:\n  "I don\'t have enough information in the provided company documents to answer this."\n- Do not guess or assume.\n- If documents partially match, answer only the supported portion.\n- Do not mention the prompt or these rules in your answer.\n- Keep the response clear, direct, and professional.\n\nUSER QUESTION:\nCan I install my own software on office laptop?\n\nCOMPANY DOCUMENTS:\n[\'Panulty is equivalent to the full day off\']\n\nANSWER:\n'


In [10]:
resp = chat_model.invoke(final_temp)

In [11]:
resp.content

"I don't have enough information in the provided company documents to answer this."

In [12]:
output

['Panulty is equivalent to the full day off']

# Prompts 

## single turn message 
here client will send only one msg and will use only that msg at the time of invocation

#### 1. static message
- we dont need to do anything here, direcly invoke `model` with user prompt

#### 2. Dynamic Message using Prompt Template
- Here we use the `PromptTemplate` to handle user input prompt with some dynamic changes
- This is used to restrict the scope of user's prompt

`
from langchain_core.prompts import PromptTemplate
`


## Multi-turn conversation
- In this case we send the list of messages history with model everytime so that model should understand last conversation.
- use for long term conversation

#### 1. static messages
- when will directly use user's msg without modifying it
- we use to append the msg share by User and responses shared by AI
- to inform about who sent which msg we use `SystemMessage, HumanMessage, AIMessage` and append these to an history array
- and at the time of invocation we will send this history with model

`
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
`

#### 2. dynamic messages
- `ChatPromptTemplate` used tp handle dynamic chat. 
- `MessagesPlaceholder` used to handle to load previous chat history which were stored in the DB. 
- we will retrive those histories and put then inside this `MessagePlaceHolder`.
- this `MessagesPlaceholder` use inside `ChatPromptTemplate` at the time of instance creation

`
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
`

In [13]:
from langchain_core.prompts import  PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [14]:
from langchain_core.prompts import MessagesPlaceholder
saved_chat_history = MessagesPlaceholder(variable_name = "stored_chats")
saved_chat_history.format_messages(
    stored_chats=[
        ("system", "You are an AI assistant."),
        ("human", "Hello!"),
    ]
)

[SystemMessage(content='You are an AI assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello!', additional_kwargs={}, response_metadata={})]

### structured Output
```
structured_model = model.with_structured_output(output_schema)

structured_model.invoke(input_msg)
```

In [15]:
from pydantic import BaseModel, Field

class States(BaseModel):
    capital: str = Field(description="capital of any country of state")



In [16]:
structured_model = chat_model.with_structured_output(States)
output = structured_model.invoke("what is the capital of india")

In [17]:
print(output.model_dump())

{'capital': 'New Delhi'}


In [18]:
chain = template | chat_model

In [21]:
# draw graph
chain.get_graph().print_ascii()

  +-------------+    
  | PromptInput |    
  +-------------+    
          *          
          *          
          *          
 +----------------+  
 | PromptTemplate |  
 +----------------+  
          *          
          *          
          *          
   +------------+    
   | ChatOllama |    
   +------------+    
          *          
          *          
          *          
+------------------+ 
| ChatOllamaOutput | 
+------------------+ 


In [20]:
print(template)

input_variables=['query_input', 'similar_docs'] input_types={} partial_variables={} template='\nYou are a company internal assistant. Your job is to answer employee questions strictly using ONLY the provided context from company_docs.\n\nRULES:\n- Use ONLY the information present in company_docs.\n- Do NOT use prior knowledge or external information.\n- If the answer is not clearly present, say:\n  "I don\'t have enough information in the provided company documents to answer this."\n- Do not guess or assume.\n- If documents partially match, answer only the supported portion.\n- Do not mention the prompt or these rules in your answer.\n- Keep the response clear, direct, and professional.\n\nUSER QUESTION:\n{query_input}\n\nCOMPANY DOCUMENTS:\n{similar_docs}\n\nANSWER:\n'


In [ ]:
chain.invoke

# Document Loader

**Document Loaders** are usually used to load a lot of Documents in a single run.

we need to import `loaders` from `langchain_community.document_loader`
there are 100s of document loader available



### steps to load a doc:
- create the object of that document loader by providing paths and encoding
- call `load()` or `lazy_load()` method of that object
- it will return object of array where each object contains metadata = {} and page_content="...."

**load()** is used when you wants to load a single doc or all the docs at once. 
- it will return a `[{metadata={page}, page_content="...."}]`
- loads all the document at once in the memory
- very costly 

**lazy_load()** is used when you don't wants to load all the documents in the memory at once.
- used when we need to load multiple doc
- it will load one document at a time
- it returns the generator so that we can load other document one-by-one whenever we need
- `DirectoryLoader`



### Document loaders

*load web pages*
- **WebBaseLoader**: load static web pages.
- **SeleniumURLLoader**: loads javascript heavy web pages

*load files*
- **TextLoader**: load text files. it will return a single objent instide the response array, coz the full text file will be treated as a signle doc
- **CSVLoader**: each row treated as new page 
- **PyPDFLoader**: load PDF files 

*Directory loader*
- **DirectoryLoader**: load the multiple files present in teh directory

***Custom loader***

Inherit the `BaseLoader` to create the custom loader 




In [ ]:
# !pip install langchain-community

from langchain_community.document_loaders import TextLoader, DirectoryLoader, CSVLoader

loader = TextLoader(
    file_path="dumps/docs/text-doc.txt",
    encoding="utf-8"
)



In [11]:
text_loader_res = loader.load()
# text_loader_res = loader.load()
# response [(metadata= {}, page_content ="....")]


In [9]:
text_loader_res

[Document(metadata={'source': 'docs/text-doc.txt'}, page_content='Test doc file\n\nnothing is here\n')]

# Text Splitter

#### Charactor based text splitter (rearly used) 
- not efficent coz it breaks the words
`CharacterTextSplitter()`


#### Recursive text based text splitter 
- mostly use, more efficient
- it recursively try to split and find the maximum safe grouping of words which called tokens
- it try with different splitting patterns: "\n\n", "\n", " ", ""

`RecursiveCharacterTextSplitter()`


#### Document structure based 
- it is used for special kind of docuemnts like code files or markdown etc. 
- it is the extension of recursive text splitter. 
- here just we will used different splitter pattern
```
RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size, chunk_overlap
)

```

#### Semantic meaning based splitting
this type of splitting used when we have multiple paragraphs and we want to split based on their meaning.
- here first we split entire document into piece of sentances, and generate their embeddings. 
- we use the sliding window and match the similarity between two consicative sentences,
- if similarity is high then the *THRESHOLD* then we group them and consider them as a part of same token
- else will get to know that similarity is less so this sentence is having different topic, so we split them from here 
this is a new technique and required more llm computation and still in testing face 


In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language


splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, 
    chunk_size=50, # max chars in each chunk
    chunk_overlap=0 # this is optional, indicate how much chars overlap you want between chunks
)

python_code = """
def example_function():
    print("This is an example.")
    for i in range(5):
        print(i)    
        
example_function()
"""

splitter.split_text(python_code)

['def example_function():',
 'print("This is an example.")',
 'for i in range(5):\n        print(i)',
 'example_function()']

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
test_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

inputs = """LangChain is a framework for developing applications powered by language models. It can be used for chatbots, Generative Question-Answering (GQA), summarization, and much more.
The core idea of the library is that we can "chain" together different components to create more advanced use cases around LLMs. Chains may consist of multiple components from several modules, or just a single component. The most important module is the llms module which provides a standard interface for all LLMs, and contains wrappers for all major providers.
The other modules are agents, which include implementations of various agent types, and memory, which includes implementations of various memory types. The document_loaders module includes a large number of utilities for loading documents in many different formats, and the text_splitters module includes a large number of utilities for splitting text into chunks.
"""

test_splitter.split_text(inputs)


['LangChain is a framework for developing',
 'applications powered by language models. It can',
 'It can be used for chatbots, Generative',
 'Question-Answering (GQA), summarization, and much',
 'and much more.',
 'The core idea of the library is that we can',
 'we can "chain" together different components to',
 'to create more advanced use cases around LLMs.',
 'LLMs. Chains may consist of multiple components',
 'from several modules, or just a single component.',
 'The most important module is the llms module',
 'module which provides a standard interface for',
 'for all LLMs, and contains wrappers for all major',
 'all major providers.',
 'The other modules are agents, which include',
 'include implementations of various agent types,',
 'types, and memory, which includes implementations',
 'of various memory types. The document_loaders',
 'module includes a large number of utilities for',
 'for loading documents in many different formats,',
 'formats, and the text_splitters module i

# Vector Store 
A vector store is a system design to store and retrive data represnted as numerical vectors.

#### key features
1. storage: ensure that vectors and treir associated metadata are retained, whether **in-memory** for quick lookups or **on-disk** for durability and large-scale use.
2. Similarity search: helps retrieve the vectors mostsimilar to a query vector
3. **Indexing**: Provide a data structure or method taht enables fast similarity seraches on high-dimensional vectors (e.g., **approximate nearest neighbor lookups**)
4. CRUD Operations: manage the lifecycle of data--addeing new vectors, reading them, updating existing entries, removing outdated vectors

#### Use- cases
1. sentic search
2. RAG
3. Recommender systems
4. Image/Multimedia search


## Vector Store VS Vector database
### Vector Store
- Typically refers to a lightweight library or service that focuses on storing vectors (embeddings) and performing similarity search.
- May not include many traditional database features like transactions, rich query languages, or role-based access control.
- Ideal for prototyping, smaller-scale applications.
- Examples: *FAISS* (a facebook vector lib, where you store vectors and can query them by similarity, but you handle persistence and scaling separately).

### Vector Database
- Definition: A full-fledged database system designed to store and query vectors.
- Additional "Database-like" Features:
    - Distributed architecture for horizontal scaling.
    - Durability and persistence (replication, backup/restore).
    - Metadata handling (schemas, filters).
    - Potential for ACID or near-ACID guarantees.
    - Authentication/authorization and more advanced security.
- Best Use Case: Geared for production environments with significant scaling and large datasets.


### System Comparison (Handwritten Notes)
- A vector Database is effectively a vector store with extra database features (e.g., clustering, scaling, security, metadata filtering, and durability)'
- all vector databases are vector store, but not all vector store are vector databases.

**Vector Store (Lightweight System)**
- Storage
- Retrieval

**Vector Database (Enterprise System)**
- Distributed
- Backup and restore
- ACID Trans (Transactions)
- Concurrency
- Authentical (Authentication/Security)




In [ ]:
from langchain_community.vectorstores import Qdrant

client = Qdrant()

In [ ]:
from langchain_core.output_parsers import pydantic

# Retrievers
- we can direcly use vector store as retrivar and do `similarity_search` but the restrictition is it will do a very basic `similarity_search`
- if we want to do some advanced type of searching stretagy then we need **Retrivals**

`retrival = vectorstore.as_retriever( retriever_type = "sementic_search")` // convert to retriever

### Types of retrievers

1. Maximal Marginal Relevence (MMR)


2. Multi-Query Retriever:
    - Initialized Base Retrived(some vectorDB, LLM)
    - generate multiple different relevent query using LLM
    - each query use to retrieve some documents from vector db 
    - combine all the retrieved docs retrieved using different LLM generated queries
    - return the final result

    - **use Case:** This will be useful when we have ambigity in query. means used doesn't provide clearance of in the query


3. Contextual Compression Retrievar: 
    - Initialized Base retriever(some vectorDB) 
    - retrieved docs from vector DB
    - compressor (uses LLM and applied to all the retrieved docs ) 
    - compressor (keep relevent part from the retrieved docs) 
    - descard irrelevent part from the retrived docs

    - **use case:** this will be useful when we have the large document which holds mutiple meanings. stored Large doc with multiple things in single doc and you only want relevent information from it

##### Other Retriever
4. ParentDocumentRetriever
5. TimeWeightedVectorRetriever
6. SelfQueryRetriever
7. MultiVectorRetriever 
8. EnsembleRetriever